In [1]:
import pandas as pd
import joblib
from pyprojroot import here
from features import transform_features

test_df = pd.read_parquet(
    here("data/processed/test.parquet")
)
val_df = pd.read_parquet(
    here("data/processed/val.parquet")
)

In [2]:
model = joblib.load(
    here("models/xgb_final.pkl")
)

encoders = joblib.load(
    here("models/encoders.pkl")
)

In [3]:
X_test = transform_features(
    test_df,
    encoders
)

y_test = test_df["DepDel15"]

X_val = transform_features(
    val_df,
    encoders
)

y_val = val_df["DepDel15"]

In [4]:
test_probs = model.predict_proba(X_test)[:, 1]

test_preds = (test_probs >= 0.5).astype(int)

In [5]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

pr_auc = average_precision_score(
    y_test,
    test_probs
)

roc_auc = roc_auc_score(
    y_test,
    test_probs
)

print(f"Test PR-AUC : {pr_auc:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")

Test PR-AUC : 0.3270
Test ROC-AUC: 0.6538


In [6]:
print(
    classification_report(
        y_test,
        test_preds
    )
)

              precision    recall  f1-score   support

         0.0       0.78      1.00      0.88    458558
         1.0       0.47      0.01      0.01    126274

    accuracy                           0.78    584832
   macro avg       0.63      0.50      0.45    584832
weighted avg       0.72      0.78      0.69    584832



In [7]:
cm = confusion_matrix(
    y_test,
    test_preds
)

print(cm)

[[457624    934]
 [125435    839]]


In [8]:
results = pd.DataFrame({
    "actual": y_test,
    "pred_prob": test_probs,
    "pred_label": test_preds
})

results.to_csv(
    here("models/test_predictions.csv"),
    index=False
)

In [11]:
from sklearn.dummy import DummyClassifier

train_df = pd.read_parquet(
    here("data/processed/train.parquet")
)
X_train = transform_features(
    train_df,
    encoders
)

y_train = train_df["DepDel15"]

# December's actual delay rate = the no-skill PR-AUC floor for the test set
print("December delay rate (test floor):", y_test.mean())
print("November delay rate (val floor):", y_val.mean())

# Dummy floor on December, computed properly
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
dummy_test = average_precision_score(y_test, dummy.predict_proba(X_test)[:, 1])

# The test set MUST be transformed with the SAME train-fit encoders,
# never refit on test data.
enc = joblib.load(here("models/encoders.pkl"))
X_test_check = transform_features(test_df, enc)   # train-fit encoders only

# A few test rows' encoded values should match the TRAIN mapping exactly
print(X_test_check[["airline_delay_rate","origin_delay_rate","dest_delay_rate"]].describe())
print("global_mean in encoders:", enc["global_mean"])

December delay rate (test floor): 0.21591499781133727
November delay rate (val floor): 0.14377383364834692
       airline_delay_rate  origin_delay_rate  dest_delay_rate
count       584832.000000      584832.000000    584832.000000
mean             0.171824           0.172935         0.172893
std              0.036595           0.033372         0.024160
min              0.099753           0.073034         0.078341
25%              0.143069           0.145637         0.156885
50%              0.180727           0.170619         0.173059
75%              0.185886           0.196701         0.185404
max              0.252543           0.284106         0.309494
global_mean in encoders: 0.17230507420387375
